### Valid WASM

In [ ]:
def runwasm(wasmfile):
    from IPython.display import display, Javascript
    with open(wasmfile, "rb") as file: 
        wasm_bytes = file.read() # read the WASM file and convert its content to a comma-separated string of byte values
        wasm_byte_array_str = ','.join(str(byte) for byte in wasm_bytes)
 
    display(Javascript(f""" // Javascript code to compile and run the WASM module
    const wasmBinary = new Uint8Array([{wasm_byte_array_str}]); // convert back to binary representation
 
    const params = {{
        'P0lib': {{
            'write': i => element.append(i + ' '),
            'writeln': () => element.append(document.createElement('br')),
            'read': () => window.prompt()
        }}
    }};
 
    WebAssembly.compile(wasmBinary.buffer) // compile shareable code
        .then(module => WebAssembly.instantiate(module, params)) // create an instance with memory
        // .then(instance => instance.exports.program()); // run the main program; not needed if a start function is specified
    """))

In [ ]:
def runpywasm(wasmfile):
    from pywasm.core import Machine, Runtime, FuncType, ValType
    # P0lib implementation in Python
    def write(_: Machine, args: list[int]) -> list[int]:
        print(args[0]); return []
    def writeln(_: Machine, args: list[int]) -> list[int]:
        print(); return []
    def read(_: Machine, args: list[int]) -> list[int]:
        return [int(input())]
    # Create runtime
    runtime = Runtime()
    runtime.imports = {'P0lib':
        {'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
         'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
         'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read)}}
    # Create and run instance
    instance = runtime.instance_from_file(wasmfile)

In [ ]:
def runwasmtime(wasmfile):
    from wasmtime import Engine, Store, Module, Linker, Func, FuncType, ValType
    # P0lib implementation in Python
    def write(i: int): print(i)
    def writeln(): print()
    def read() -> int: return int(input())
    # Create engine, store, and module
    engine = Engine()
    store = Store(engine)
    module = Module(store.engine, open(wasmfile, 'rb').read())
    # Use Linker to create the P0lib library
    write_func = Func(store, FuncType([ValType.i32()], []), write)
    writeln_func = Func(store, FuncType([], []), writeln)
    read_func = Func(store, FuncType([], [ValType.i32()]), read)
    linker = Linker(engine)
    linker.define(store, "P0lib", "write", write_func)
    linker.define(store, "P0lib", "writeln", writeln_func)
    linker.define(store, "P0lib", "read", read_func)
    # Create and run an instance
    instance = linker.instantiate(store, module)

Consider the following WebAssembly programs. For each, argue if it is valid or not (i.e. produces an error when being loaded), if it traps at run-time, or if it works without error. In the last case, argue what it does. Use `!wat2wasm ...`, `runwasm(...)`, `runpywasm(...)`, and `runwasmtime(...)` to check your answer! 

In [ ]:
%%writefile a.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (func $program
    i32.const 3
    i32.const 4
    i32.add
    i32.add
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm a.wat

In [ ]:
%%writefile b.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (global $x (mut i32) i32.const 3)
  (global $y (mut i32) i32.const 5)
  (func $program
    global.get $x
    global.get $y
    i32.gt_s
    if
      global.get $x
      call $write
    else
      global.get $y
    end
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm b.wat

In [ ]:
%%writefile c.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (global $x (mut i32) i32.const 3)
  (global $y (mut i32) i32.const 5)
  (func $program
    global.get $x
    global.get $y
    i32.gt_s
    if (result i32)
      global.get $x
    else
      global.get $y
    end
    call $write
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm c.wat

In [ ]:
runwasm("c.wasm")

In [ ]:
runpywasm("c.wasm")

In [ ]:
runwasmtime("c.wasm")

In [ ]:
%%writefile d.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (global $x (mut i32) i32.const 5)
  (global $y (mut i32) i32.const 0)
  (func $program
    global.get $x
    global.get $y
    i32.div_s
    call $write
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm d.wat

In [ ]:
runwasm("d.wasm")

In [ ]:
runpywasm("d.wasm")

In [ ]:
runwasmtime("d.wasm")

In [ ]:
%%writefile e.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (func $program
    i32.const 0
    i32.const 3
    i32.store offset=0
    i32.const 4
    i32.const 5
    i32.store offset=0
    i32.const 0
    i32.load offset=0
    i32.const 0
    i32.load offset=4
    i32.add
    call $write
  )
  (memory 1)
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm e.wat
runwasm("e.wasm")

In [ ]:
runpywasm("e.wasm")

In [ ]:
runwasmtime("e.wasm")

In [ ]:
%%writefile f.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (func $program
    i32.const 0
    i32.const 3
    i32.store offset=0
    i32.const 0
    i32.const 5
    i32.store offset=4
    i32.const -4
    i32.load offset=0
    i32.const 4
    i32.load offset=0
    i32.add
    call $write
  )
  (memory 1)
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm f.wat
runwasm("f.wasm")

In [ ]:
runpywasm("f.wasm")

In [ ]:
runwasmtime("f.wasm")

In [ ]:
%%writefile g.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (func $max (param $x i32) (param $y i32) (result i32)
    (local $m i32)
    local.get $x
    local.get $y
    i32.gt_s
    if
      local.get $x
      local.set $m
    else
      local.get $y
      local.set $m
    end
    local.get $m
  )
  (func $program
    i32.const 3
    i32.const 5
    call $max
    call $write
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm g.wat

In [ ]:
runwasm("g.wasm")

In [ ]:
runpywasm("g.wasm")

In [ ]:
runwasmtime("g.wasm")

In [ ]:
%%writefile h.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (func $max (param $x i32) (param $y i32) (result i32)
    (local $m i32)
    local.get $x
    local.get $y
    i32.gt_s
    if
      local.get $x
      local.set $m
    else
      local.get $y
      local.set $m
    end
    local.get $m
  )
  (func $program
    i32.const 3
    call $max
    call $write
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm h.wat

In [ ]:
%%writefile i.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (func $max (param $x i32) (param $y i32) (result i32)
    (local $m i32)
    local.get $x
    local.get $y
    i32.gt_s
    if
      local.get $x
      local.set $m
    else
      local.get $y
      local.set $m
    end
  )
  (func $program
    i32.const 3
    i32.const 5
    call $max
    call $write
    i32.const 9
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm i.wat

In [ ]:
%%writefile j.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (func $program
    (local $x i32)
    block
      i32.const 5
      local.set $x
      br 0
      i32.const 7
      local.set $x
    end
    local.get $x
    call $write
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm j.wat

In [ ]:
runwasm("j.wasm")

In [ ]:
runpywasm("j.wasm")

In [ ]:
runwasmtime("j.wasm")

In [ ]:
%%writefile k.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (func $program
    (local $x i32)
    block
      i32.const 5
      local.set $x
      br 1
      i32.const 7
      local.set $x
    end
    local.get $x
    call $write
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm k.wat

In [ ]:
runwasm("k.wasm")

In [ ]:
runpywasm("k.wasm")

In [ ]:
runwasmtime("k.wasm")

In [ ]:
%%writefile l.wat
(module
  (import "P0lib" "write" (func $write (param i32)))
  (func $program
    (local $x i32)
    block
      i32.const 5
      local.set $x
      br 2
      i32.const 7
      local.set $x
    end
    local.get $x
    call $write
  )
  (start $program)
)

YOUR ANSWER HERE

In [ ]:
!wat2wasm l.wat